In [1]:
import pandas as pd
import numpy as np
import os

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

print("Libraries loaded successfully")

Libraries loaded successfully


Load .env

In [2]:
load_dotenv()

MYSQL_HOST = os.getenv("MYSQL_HOST")
MYSQL_PORT = os.getenv("MYSQL_PORT")
MYSQL_USER = os.getenv("MYSQL_USER")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD")
MYSQL_DATABASE = os.getenv("MYSQL_DATABASE")

print("Host:", MYSQL_HOST)
print("Port:", MYSQL_PORT)
print("User:", MYSQL_USER)
print("Database:", MYSQL_DATABASE)

Host: localhost
Port: 3306
User: root
Database: agentic_analytics


Create MySQL connection

In [3]:
connection_url = (
    f"mysql+mysqlconnector://"
    f"{MYSQL_USER}:{MYSQL_PASSWORD}@"
    f"{MYSQL_HOST}:{MYSQL_PORT}/"
    f"{MYSQL_DATABASE}"
)

engine = create_engine(connection_url)

print("MySQL engine created successfully")

MySQL engine created successfully


Then test the connection:

In [4]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT DATABASE();"))
    print("Connected database:", result.scalar())

Connected database: agentic_analytics


Load our processed CSV files

In [5]:
PROCESSED_PATH = "../data/processed/"

In [6]:
customers = pd.read_csv(
    f"{PROCESSED_PATH}customers_clean.csv"
)

orders = pd.read_csv(
    f"{PROCESSED_PATH}orders_analytics.csv"
)

order_items = pd.read_csv(
    f"{PROCESSED_PATH}order_items_clean.csv"
)

payments = pd.read_csv(
    f"{PROCESSED_PATH}payments_clean.csv"
)

reviews = pd.read_csv(
    f"{PROCESSED_PATH}reviews_clean.csv"
)

products = pd.read_csv(
    f"{PROCESSED_PATH}products_clean.csv"
)

sellers = pd.read_csv(
    f"{PROCESSED_PATH}sellers_clean.csv"
)

print("All processed datasets loaded.")

All processed datasets loaded.


In [7]:
print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Order Items:", order_items.shape)
print("Payments:", payments.shape)
print("Reviews:", reviews.shape)
print("Products:", products.shape)
print("Sellers:", sellers.shape)

Customers: (99441, 5)
Orders: (99441, 22)
Order Items: (112650, 7)
Payments: (103886, 5)
Reviews: (99224, 7)
Products: (32951, 11)
Sellers: (3095, 4)


We need to check the actual column names in these processed CSVs against our MySQL schema.

In [8]:
print("CUSTOMERS")
print(customers.columns.tolist())

print("\nORDERS")
print(orders.columns.tolist())

print("\nORDER ITEMS")
print(order_items.columns.tolist())

print("\nPAYMENTS")
print(payments.columns.tolist())

print("\nREVIEWS")
print(reviews.columns.tolist())

print("\nPRODUCTS")
print(products.columns.tolist())

print("\nSELLERS")
print(sellers.columns.tolist())

CUSTOMERS
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

ORDERS
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_days', 'delivery_delay_days', 'is_late', 'product_revenue', 'freight_cost', 'total_items', 'total_order_value', 'order_date', 'order_year', 'order_month', 'order_month_name', 'order_quarter', 'order_day', 'order_day_of_week']

ORDER ITEMS
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

PAYMENTS
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

REVIEWS
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

PRODUCTS
['product_id', 'product_category_name', 'product_name_lenght', 'product

In [9]:
print("CUSTOMERS")
print(customers.columns.tolist())

print("\nORDERS")
print(orders.columns.tolist())

print("\nORDER ITEMS")
print(order_items.columns.tolist())

CUSTOMERS
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

ORDERS
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_days', 'delivery_delay_days', 'is_late', 'product_revenue', 'freight_cost', 'total_items', 'total_order_value', 'order_date', 'order_year', 'order_month', 'order_month_name', 'order_quarter', 'order_day', 'order_day_of_week']

ORDER ITEMS
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']


Prepare Data Before MySQL Insert

In [10]:
# =========================================================
# Prepare datasets for MySQL
# =========================================================

# Remove order_date because it is not part of the MySQL orders table
orders_mysql = orders.drop(columns=["order_date"])

# Convert datetime columns
order_datetime_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in order_datetime_cols:
    orders_mysql[col] = pd.to_datetime(
        orders_mysql[col],
        errors="coerce"
    )

# Order items datetime
order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"],
    errors="coerce"
)

# Payments doesn't normally require datetime conversion
# Reviews datetime columns
review_datetime_cols = [
    "review_creation_date",
    "review_answer_timestamp"
]

for col in review_datetime_cols:
    reviews[col] = pd.to_datetime(
        reviews[col],
        errors="coerce"
    )

print("Data preparation completed.")

Data preparation completed.


Check the prepared columns

In [11]:
print("Customers:", customers.shape)
print("Orders:", orders_mysql.shape)
print("Order Items:", order_items.shape)
print("Payments:", payments.shape)
print("Reviews:", reviews.shape)
print("Products:", products.shape)
print("Sellers:", sellers.shape)

Customers: (99441, 5)
Orders: (99441, 21)
Order Items: (112650, 7)
Payments: (103886, 5)
Reviews: (99224, 7)
Products: (32951, 11)
Sellers: (3095, 4)


In [12]:
print("\nOrders columns:")
print(orders_mysql.columns.tolist())


Orders columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_days', 'delivery_delay_days', 'is_late', 'product_revenue', 'freight_cost', 'total_items', 'total_order_value', 'order_year', 'order_month', 'order_month_name', 'order_quarter', 'order_day', 'order_day_of_week']


Load in the correct order

Before loading, verify all remaining datasets

In [13]:
print("PAYMENTS")
print(payments.columns.tolist())

print("\nREVIEWS")
print(reviews.columns.tolist())

print("\nPRODUCTS")
print(products.columns.tolist())

print("\nSELLERS")
print(sellers.columns.tolist())

PAYMENTS
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

REVIEWS
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

PRODUCTS
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english', 'product_category']

SELLERS
['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']


Load all 9 tables into MySQL

Load geolocation and translation

In [15]:
# =========================================================
# Define project paths
# =========================================================

RAW_PATH = "../data/raw/"
PROCESSED_PATH = "../data/processed/"

print("Raw path:", RAW_PATH)
print("Processed path:", PROCESSED_PATH)

Raw path: ../data/raw/
Processed path: ../data/processed/


In [16]:
# =========================================================
# Load remaining raw datasets
# =========================================================

geolocation = pd.read_csv(
    f"{RAW_PATH}olist_geolocation_dataset.csv"
)

category_translation = pd.read_csv(
    f"{RAW_PATH}product_category_name_translation.csv"
)

print("Geolocation:", geolocation.shape)
print("Category Translation:", category_translation.shape)

Geolocation: (1000163, 5)
Category Translation: (71, 2)


Prepare geolocation

In [17]:
# =========================================================
# Prepare geolocation
# =========================================================

geolocation = geolocation.drop_duplicates()

print("Geolocation after removing exact duplicates:")
print(geolocation.shape)

Geolocation after removing exact duplicates:
(738332, 5)


Prepare category translation

In [18]:
# =========================================================
# Prepare category translation
# =========================================================

category_translation = category_translation.drop_duplicates()

print("Category translation:")
print(category_translation.shape)

Category translation:
(71, 2)


Create a loading function

In [19]:
# =========================================================
# MySQL ETL Helper
# =========================================================

def load_to_mysql(df, table_name):
    """
    Load a Pandas DataFrame into a MySQL table.
    """

    print(f"Loading {table_name}...")

    df.to_sql(
        name=table_name,
        con=engine,
        if_exists="append",
        index=False,
        chunksize=5000,
        method="multi"
    )

    print(f"✓ {table_name} loaded successfully")

Load the tables

In [20]:
# =========================================================
# Load data into MySQL
# =========================================================

load_to_mysql(customers, "customers")

load_to_mysql(products, "products")

load_to_mysql(sellers, "sellers")

load_to_mysql(orders_mysql, "orders")

load_to_mysql(order_items, "order_items")

load_to_mysql(payments, "payments")

load_to_mysql(reviews, "reviews")

load_to_mysql(geolocation, "geolocation")

load_to_mysql(category_translation, "category_translation")

Loading customers...
✓ customers loaded successfully
Loading products...
✓ products loaded successfully
Loading sellers...
✓ sellers loaded successfully
Loading orders...
✓ orders loaded successfully
Loading order_items...
✓ order_items loaded successfully
Loading payments...
✓ payments loaded successfully
Loading reviews...


IntegrityError: (mysql.connector.errors.IntegrityError) 1062 (23000): Duplicate entry '3242cc306a9218d0377831e175d62fbf' for key 'reviews.PRIMARY'
[SQL: INSERT INTO reviews (review_id, order_id, review_score, review_comment_title, review_comment_message, review_creation_date, review_answer_timestamp) VALUES (%(review_id_m0)s, %(order_id_m0)s, %(review_score_m0)s, %(review_comment_title_m0)s, %(review_comment_message_m0)s, %(review_creation_date_m0)s, %(review_answer_timestamp_m0)s), (%(review_id_m1)s, %(order_id_m1)s, %(review_score_m1)s, %(review_comment_title_m1)s, %(review_comment_message_m1)s, %(review_creation_date_m1)s, %(review_answer_timestamp_m1)s), (%(review_id_m2)s, %(order_id_m2)s, %(review_score_m2)s, %(review_comment_title_m2)s, %(review_comment_message_m2)s, %(review_creation_date_m2)s, %(review_answer_timestamp_m2)s), (%(review_id_m3)s, %(order_id_m3)s, %(review_score_m3)s, %(review_comment_title_m3)s, %(review_comment_message_m3)s, %(review_creation_date_m3)s, %(review

In [21]:
# Check duplicate review IDs
duplicate_reviews = reviews[
    reviews["review_id"].duplicated(keep=False)
].sort_values("review_id")

print("Duplicate review rows:", len(duplicate_reviews))
print("Duplicate review IDs:", duplicate_reviews["review_id"].nunique())

Duplicate review rows: 1603
Duplicate review IDs: 789


In [22]:
duplicate_reviews.head(20)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09,2017-09-13 09:52:44


In [24]:
reviews["review_id"].duplicated().sum()

np.int64(0)

In [25]:
reviews = reviews.drop_duplicates(
    subset=["review_id"],
    keep="first"
)

print("Reviews after removing duplicate IDs:", reviews.shape)

Reviews after removing duplicate IDs: (98410, 7)


In [26]:
load_to_mysql(reviews, "reviews")

Loading reviews...
✓ reviews loaded successfully


In [27]:
print("Duplicate review IDs:",
      reviews["review_id"].duplicated().sum())

Duplicate review IDs: 0


In [28]:
load_to_mysql(geolocation, "geolocation")

Loading geolocation...
✓ geolocation loaded successfully


In [29]:
load_to_mysql(category_translation, "category_translation")

Loading category_translation...
✓ category_translation loaded successfully
